In [1]:
# %%
# =============================================================================
# NOTEBOOK: kSZ × v_rec Cross-Power Spectrum
# =============================================================================
#
# Standalone — loads directly from cluster paths, no export step needed.
#
# WHAT IT MEASURES
# ----------------
# C_ell( T_kSZ × v_rec )  vs  C_ell( T_kSZ × v_true )  per z-bin z=5–8
# Ratio → reconstruction penalty r — realistic SNR forecast for LAE kSZ
#
# PIPELINE PER Z-BIN
# ------------------
# 1. Paint LAE positions onto 2D grid → delta_LAE
# 2. FFT linear reconstruction → v_rec  (continuity equation)
# 3. Average kSZ integrand slices in z-bin → T_kSZ(x,y)
# 4. Average true v_los slices in z-bin   → v_true(x,y)
# 5. q_rec  = T_kSZ × v_rec
#    q_true = T_kSZ × v_true
# 6. 2D FFT power spectra → C_ell per z-bin
# 7. r = C_cross / sqrt(C_rec × C_true)  — reconstruction correlation
# =============================================================================

# %%
# =============================================================================
# CELL 1: Imports and configuration
# =============================================================================

import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import py21cmfast as p21c
from astropy.cosmology import FlatLambdaCDM

# ── cluster paths ──────────────────────────────────────────────────────────────

SIMPLEGEN_DIR  = "/user1/swanith/SiMPLE-Gen/SiMPLEGen/data"
CACHE_DIR_READ  = "/user1/swanith/kSZ2_halo_project/cache"        # read — existing data
CACHE_DIR_WRITE = "/user1/swanith/kSZ2_halo_project/ksz_vrec"     # write — new outputs
PLOT_DIR        = "/user1/swanith/kSZ2_halo_project/ksz_vrec/plots"

os.makedirs(CACHE_DIR_WRITE, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

# ── seeds ──────────────────────────────────────────────────────────────────────
RANDOM_SEEDS = list(range(1, 6))

# ── box geometry (fixed) ───────────────────────────────────────────────────────
BOX_LEN  = 400.0   # cMpc
HII_DIM  = 32
CELL_MPC = BOX_LEN / HII_DIM   # 12.5 cMpc

# ── LAE selection cuts (matching your main notebook) ──────────────────────────
REW_CUT  = 10.0    # Angstrom
LLYA_CUT = 1e42    # erg/s

# ── reconstruction parameters ──────────────────────────────────────────────────
R_SMOOTH_MPC = 10.0   # Gaussian smoothing scale [cMpc] — ~bubble scale at z~7
F_GROW       = 0.97   # linear growth rate f ~ Omega_m(z)^0.55 at z~6-8

# ── redshift bins ──────────────────────────────────────────────────────────────
Z_BINS = [(5.0, 5.5), (5.5, 6.0), (6.0, 6.5),
          (6.5, 7.0), (7.0, 7.5), (7.5, 8.0)]

# ── cosmology ──────────────────────────────────────────────────────────────────
cosmo = FlatLambdaCDM(H0=67.77, Om0=0.3086, Ob0=0.0489, Tcmb0=2.7255)

# ── plot style ─────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'serif', 'mathtext.fontset': 'cm',
    'font.size': 14, 'axes.labelsize': 14,
    'xtick.direction': 'in', 'ytick.direction': 'in',
    'xtick.top': True, 'ytick.right': True,
    'xtick.minor.visible': True, 'ytick.minor.visible': True,
    'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
})

print(f"✓ Cell 1 done")
print(f"  BOX_LEN={BOX_LEN} cMpc  HII_DIM={HII_DIM}  cell={CELL_MPC:.2f} cMpc")
print(f"  Seeds: {RANDOM_SEEDS}")
print(f"  Z bins: {Z_BINS}")

# %%

✓ Cell 1 done
  BOX_LEN=400.0 cMpc  HII_DIM=32  cell=12.50 cMpc
  Seeds: [1, 2, 3, 4, 5]
  Z bins: [(5.0, 5.5), (5.5, 6.0), (6.0, 6.5), (6.5, 7.0), (7.0, 7.5), (7.5, 8.0)]


In [2]:
# =============================================================================
# CELL 2: Load per-seed data
# =============================================================================

print("\n" + "="*70)
print("CELL 2 — Loading data from cluster paths")
print("="*70)

ksz_int_all = {}   # kSZ integrand  (HII_DIM, HII_DIM, n_z)
v_true_all  = {}   # true v_los     (HII_DIM, HII_DIM, n_lc)  in km/s
z_lc_all    = {}   # lightcone redshifts (n_lc,)
lae_all     = {}   # LAE catalogues

for seed in RANDOM_SEEDS:

    # ── kSZ integrand ─────────────────────────────────────────────────────────
    ksz_path = os.path.join(CACHE_DIR_READ, "kSZ_integrands",
                            f"kSZ_integrand_seed{seed}.npy")
    if not os.path.exists(ksz_path):
        print(f"  ✗ seed {seed}: kSZ integrand missing at {ksz_path}")
        continue
    ksz_int_all[seed] = np.load(ksz_path)   # (HII_DIM, HII_DIM, n_z)

    # ── true velocity — from field_arrays.npz ─────────────────────────────────
    fields_path = os.path.join(CACHE_DIR_READ, f"seed_{seed}", "field_arrays.npz")
    if not os.path.exists(fields_path):
        print(f"  ✗ seed {seed}: field_arrays.npz missing")
        continue
    fields = np.load(fields_path)
    # units: Mpc/s → km/s
    v_true_all[seed] = fields['los_velocity_lc'].astype(np.float32)   # keep native Mpc/s

    # ── lightcone redshifts — from lightcone.h5 ───────────────────────────────
    lc_path = os.path.join(CACHE_DIR_READ, f"seed_{seed}", "lightcone.h5")
    if not os.path.exists(lc_path):
        print(f"  ✗ seed {seed}: lightcone.h5 missing")
        continue
    lc = p21c.LightCone.from_file(lc_path, safe=False)
    z_lc_all[seed] = np.array(lc.lightcone_redshifts, dtype=np.float32)

    # ── LAE catalogue — from SiMPLEGen ────────────────────────────────────────
    lae_dir = os.path.join(SIMPLEGEN_DIR, f"seed_{seed}", "lightcone_lae")
    lae_files_ok = all(
        os.path.exists(os.path.join(lae_dir, f"{k}.npy"))
        for k in ["coords", "redshifts", "LLya", "damping", "REW"]
    )
    if not lae_files_ok:
        print(f"  ✗ seed {seed}: LAE files incomplete in {lae_dir}")
        continue

    coords    = np.load(os.path.join(lae_dir, "coords.npy"))      # (N, 3) cMpc
    redshifts = np.load(os.path.join(lae_dir, "redshifts.npy"))   # (N,)
    LLya      = np.load(os.path.join(lae_dir, "LLya.npy"))
    damping   = np.load(os.path.join(lae_dir, "damping.npy"))
    REW       = np.load(os.path.join(lae_dir, "REW.npy"))

    LLya_obs = LLya * damping
    is_lae   = (REW >= REW_CUT) & (LLya_obs >= LLYA_CUT) & (damping > 0)

    lae_all[seed] = {
        "coords":    coords,
        "redshifts": redshifts,
        "is_LAE":    is_lae,
    }

    n_lae = int(is_lae.sum())
    print(f"  ✓ seed {seed}:  "
          f"integrand={ksz_int_all[seed].shape}  "
          f"v_true={v_true_all[seed].shape}  "
          f"z_lc=[{z_lc_all[seed].min():.2f},{z_lc_all[seed].max():.2f}]  "
          f"LAEs={n_lae:,}")

print(f"\n✓ Cell 2 done  ({len(ksz_int_all)} seeds loaded)")



CELL 2 — Loading data from cluster paths


/user1/swanith/miniconda3/envs/p21c_v41/lib/python3.11/site-packages/attr/_make.py:3323: UserWarning: Resolution is likely too low for accurate evolved density fields. It is recommended that you either increase the resolution (DIM/BOX_LEN) or set the EVOLVE_DENSITY_LINEARLY flag to True. Got DIM=96, BOX_LEN=400.0, resolution=4.166666666666667 Mpc Mpc.
  v(inst, attr, value)
/user1/swanith/miniconda3/envs/p21c_v41/lib/python3.11/site-packages/attr/_make.py:3323: UserWarning: You are setting R_BUBBLE_MAX != 50 when INHOMO_RECO=True. This is non-standard (but allowed), and usually occurs upon manual update of INHOMO_RECO
  v(inst, attr, value)


  ✓ seed 1:  integrand=(32, 32, 237)  v_true=(32, 32, 239)  z_lc=[5.10,20.22]  LAEs=557,491
  ✓ seed 2:  integrand=(32, 32, 237)  v_true=(32, 32, 239)  z_lc=[5.10,20.22]  LAEs=567,206
  ✓ seed 3:  integrand=(32, 32, 237)  v_true=(32, 32, 239)  z_lc=[5.10,20.22]  LAEs=583,757
  ✓ seed 4:  integrand=(32, 32, 237)  v_true=(32, 32, 239)  z_lc=[5.10,20.22]  LAEs=561,494
  ✓ seed 5:  integrand=(32, 32, 237)  v_true=(32, 32, 239)  z_lc=[5.10,20.22]  LAEs=555,877

✓ Cell 2 done  (5 seeds loaded)


In [3]:

# %%
# =============================================================================
# CELL 3: Build z_mid for the kSZ integrand axis
# =============================================================================
# The kSZ integrand has shape (HII_DIM, HII_DIM, n_z).
# n_z comes from the ind_z mask in Cell 5 of the main notebook:
#   ind_z = where(z_lc <= Z_HEAT_MAX)
# So the integrand's third axis aligns with z_lc[ind_z].
# We reconstruct that mapping here.

print("\nCELL 3 — Reconstructing z_mid for integrand axis")

Z_HEAT_MAX = 20.0   # matches inputs.simulation_options.Z_HEAT_MAX

z_mid_all = {}   # z_mid_all[seed] shape (n_z,) — redshift of each integrand slice

for seed in ksz_int_all:
    z_lc  = z_lc_all[seed]
    n_int = ksz_int_all[seed].shape[2]

    # replicate the ind_z mask from Cell 5 of main notebook
    ind_z = np.where(z_lc <= Z_HEAT_MAX)[0]

    # the integrand was built from kSZ_int_mid = 0.5*(integrand[:,:-1]+integrand[:,1:])
    # so it has one fewer slice — use midpoints of consecutive z_lc[ind_z] values
    z_ind  = z_lc[ind_z]
    z_mid  = 0.5 * (z_ind[:-1] + z_ind[1:])

    # truncate or pad to match actual integrand depth
    if len(z_mid) >= n_int:
        z_mid_all[seed] = z_mid[:n_int]
    else:
        # edge case: pad with extrapolation
        z_mid_all[seed] = np.concatenate(
            [z_mid, np.full(n_int - len(z_mid), z_mid[-1])]
        )

    print(f"  seed {seed}: integrand n_z={n_int}  "
          f"z_mid=[{z_mid_all[seed].min():.2f},{z_mid_all[seed].max():.2f}]")

print("✓ Cell 3 done")


CELL 3 — Reconstructing z_mid for integrand axis
  seed 1: integrand n_z=237  z_mid=[5.11,19.84]
  seed 2: integrand n_z=237  z_mid=[5.11,19.84]
  seed 3: integrand n_z=237  z_mid=[5.11,19.84]
  seed 4: integrand n_z=237  z_mid=[5.11,19.84]
  seed 5: integrand n_z=237  z_mid=[5.11,19.84]
✓ Cell 3 done


In [4]:
# %%
# =============================================================================
# CELL 4: Helper functions
# =============================================================================

def paint_lae_density(coords_lae, HII_DIM, BOX_LEN):
    """
    Paint LAE (x,y) positions onto a 2D grid.
    Returns delta_LAE = n/n_mean - 1, shape (HII_DIM, HII_DIM).
    """
    cell  = BOX_LEN / HII_DIM
    grid  = np.zeros((HII_DIM, HII_DIM), dtype=np.float64)
    xi    = np.clip((coords_lae[:, 0] / cell).astype(int), 0, HII_DIM - 1)
    yi    = np.clip((coords_lae[:, 1] / cell).astype(int), 0, HII_DIM - 1)
    np.add.at(grid, (xi, yi), 1.0)
    mean_n = grid.mean()
    return (grid / mean_n - 1.0).astype(np.float32) if mean_n > 0 \
           else grid.astype(np.float32)
    
def reconstruct_velocity(delta_lae, BOX_LEN, z_center,
                         f_grow=0.97, R_smooth=10.0, cosmo=None):
    """
    2D linear velocity reconstruction from delta_LAE via FFT.
    Solves: v_rec(k) = i * aHf * (k_x/k^2) * W_G(k) * delta(k)
    Returns v_rec in km/s, shape (HII_DIM, HII_DIM).

    NOTE: k_x is used as the LOS direction. This matches the x-axis
    projection convention. Adjust to k_y if your LOS is along y.
    """
    from astropy import units as u

    N    = delta_lae.shape[0]
    dk   = 2.0 * np.pi / BOX_LEN
    kx   = np.fft.fftfreq(N, d=1.0 / N) * dk
    ky   = np.fft.fftfreq(N, d=1.0 / N) * dk
    KX, KY = np.meshgrid(kx, ky, indexing='ij')
    K2   = KX**2 + KY**2
    K2[0, 0] = 1.0

    W_G = np.exp(-0.5 * (np.sqrt(K2) * R_smooth)**2)

    H_z  = cosmo.H(z_center).to(u.km / u.s / u.Mpc).value
    a    = 1.0 / (1.0 + z_center)
    aHf  = a * (H_z * a) * f_grow
    delta_k = np.fft.fft2(delta_lae)
    v_rec_k = (1j * KX / K2) * W_G * delta_k * aHf
    v_rec = np.real(np.fft.ifft2(v_rec_k)).astype(np.float32)
    v_rec = v_rec / (3.086e19 / 1e3)
    return v_rec


def cross_power_2d(field1, field2, BOX_LEN, n_k_bins=20):
    """
    2D cross-power spectrum of two real fields via FFT.
    Returns k [cMpc^-1], C_k, D_k = k^2/(2pi) * C_k, n_modes.
    """
    N    = field1.shape[0]
    dk   = 2.0 * np.pi / BOX_LEN
    kx   = np.fft.fftfreq(N, d=1.0 / N) * dk
    ky   = np.fft.fftfreq(N, d=1.0 / N) * dk
    KX, KY = np.meshgrid(kx, ky, indexing='ij')
    K    = np.sqrt(KX**2 + KY**2)

    k_min   = dk
    k_max   = np.sqrt(2) * (N // 2) * dk
    k_edges = np.logspace(np.log10(k_min), np.log10(k_max), n_k_bins + 1)
    k_cents = 0.5 * (k_edges[:-1] + k_edges[1:])

    pix_area = (BOX_LEN / N)**2
    F1 = np.fft.fft2(field1) * pix_area
    F2 = np.fft.fft2(field2) * pix_area
    cross = np.real(F1 * np.conj(F2)) / BOX_LEN**2

    C_k     = np.zeros(n_k_bins)
    n_modes = np.zeros(n_k_bins, dtype=int)

    for i in range(n_k_bins):
        mask = (K >= k_edges[i]) & (K < k_edges[i + 1])
        n_modes[i] = mask.sum()
        if n_modes[i] > 0:
            C_k[i] = cross[mask].mean()

    D_k = k_cents**2 / (2.0 * np.pi) * C_k

    return k_cents, C_k, D_k, n_modes


print("✓ Cell 4: helper functions defined")


✓ Cell 4: helper functions defined


In [5]:
# %%
# =============================================================================
# CELL 5: Main loop — C_ell per z-bin per seed
# =============================================================================

print("\n" + "="*70)
print("CELL 5 — Computing C_ell(T_kSZ × v) per z-bin")
print("="*70)

# results[seed][z_label] = dict of arrays
results = {seed: {} for seed in RANDOM_SEEDS}

for seed in ksz_int_all:
    print(f"\n── seed {seed} ──")

    ksz_int  = ksz_int_all[seed]     # (32, 32, n_z)
    v_true   = v_true_all[seed]      # (32, 32, n_lc)  km/s
    z_lc     = z_lc_all[seed]        # (n_lc,)
    z_mid    = z_mid_all[seed]       # (n_z,)
    lae      = lae_all[seed]

    coords_lae    = lae["coords"][lae["is_LAE"]]      # (N_lae, 3)
    redshifts_lae = lae["redshifts"][lae["is_LAE"]]   # (N_lae,)

    for (z_lo, z_hi) in Z_BINS:
        z_center = 0.5 * (z_lo + z_hi)
        z_label  = f"z{z_lo:.1f}-{z_hi:.1f}"

        # ── LAEs in this bin ───────────────────────────────────────────────
        lae_mask  = (redshifts_lae >= z_lo) & (redshifts_lae < z_hi)
        n_lae_bin = lae_mask.sum()

        if n_lae_bin < 20:
            print(f"  {z_label}: {n_lae_bin} LAEs — too few, skipping")
            continue

        # ── kSZ integrand slices in this bin ──────────────────────────────
        int_mask = (z_mid >= z_lo) & (z_mid < z_hi)
        if int_mask.sum() == 0:
            print(f"  {z_label}: no integrand slices — skipping")
            continue
        T_ksz_2d = ksz_int[:, :, int_mask].mean(axis=2)   # (32, 32)

        # ── true velocity slices in this bin ──────────────────────────────
        v_mask = (z_lc >= z_lo) & (z_lc < z_hi)
        if v_mask.sum() == 0:
            print(f"  {z_label}: no v_true slices — skipping")
            continue
        v_true_2d = v_true[:, :, v_mask].mean(axis=2)     # (32, 32) km/s

        # ── step 1: v_rec from LAE density ────────────────────────────────
        delta_lae = paint_lae_density(coords_lae[lae_mask], HII_DIM, BOX_LEN)
        v_rec_2d  = reconstruct_velocity(
            delta_lae, BOX_LEN, z_center,
            f_grow=F_GROW, R_smooth=R_SMOOTH_MPC, cosmo=cosmo
        )   # (32, 32) km/s

        # ── step 2: momentum fields ────────────────────────────────────────
        q_rec  = T_ksz_2d * v_rec_2d    # T_kSZ × v_rec
        q_true = T_ksz_2d * v_true_2d   # T_kSZ × v_true

        # ── step 3: power spectra ──────────────────────────────────────────
        k, C_auto_rec,   D_auto_rec,   nm = cross_power_2d(q_rec,  q_rec,  BOX_LEN)
        _, C_auto_true,  D_auto_true,  _  = cross_power_2d(q_true, q_true, BOX_LEN)
        _, C_cross,      D_cross,      _  = cross_power_2d(q_rec,  q_true, BOX_LEN)

        # ── reconstruction correlation r ───────────────────────────────────
        with np.errstate(invalid='ignore', divide='ignore'):
            r = C_cross / np.sqrt(np.abs(C_auto_rec * C_auto_true))

        r_med = float(np.nanmedian(r[nm > 2]))

        results[seed][z_label] = {
            'k':          k,
            'C_rec':      C_auto_rec,
            'C_true':     C_auto_true,
            'C_cross':    C_cross,
            'D_rec':      D_auto_rec,
            'D_true':     D_auto_true,
            'D_cross':    D_cross,
            'r':          r,
            'n_modes':    nm,
            'n_lae':      int(n_lae_bin),
            'n_slices':   int(int_mask.sum()),
            'z_center':   z_center,
        }

        print(f"  {z_label}: {n_lae_bin:,} LAEs  "
              f"{int_mask.sum()} slices  r_median={r_med:.3f}")

print("\n✓ Cell 5 done")



CELL 5 — Computing C_ell(T_kSZ × v) per z-bin

── seed 1 ──
  z5.0-5.5: 263,695 LAEs  16 slices  r_median=-0.119
  z5.5-6.0: 207,009 LAEs  18 slices  r_median=0.201
  z6.0-6.5: 77,125 LAEs  16 slices  r_median=-0.181
  z6.5-7.0: 9,387 LAEs  15 slices  r_median=0.048
  z7.0-7.5: 275 LAEs  13 slices  r_median=0.050
  z7.5-8.0: 0 LAEs — too few, skipping

── seed 2 ──
  z5.0-5.5: 263,022 LAEs  16 slices  r_median=-0.216
  z5.5-6.0: 211,042 LAEs  18 slices  r_median=-0.152
  z6.0-6.5: 82,625 LAEs  16 slices  r_median=0.002
  z6.5-7.0: 10,269 LAEs  15 slices  r_median=-0.094
  z7.0-7.5: 248 LAEs  13 slices  r_median=-0.150
  z7.5-8.0: 0 LAEs — too few, skipping

── seed 3 ──
  z5.0-5.5: 266,583 LAEs  16 slices  r_median=-0.198
  z5.5-6.0: 213,453 LAEs  18 slices  r_median=0.050
  z6.0-6.5: 91,609 LAEs  16 slices  r_median=-0.190
  z6.5-7.0: 11,818 LAEs  15 slices  r_median=0.105
  z7.0-7.5: 289 LAEs  13 slices  r_median=-0.116
  z7.5-8.0: 5 LAEs — too few, skipping

── seed 4 ──
  z5.0-5.5

In [12]:
seed = 1
z_lo, z_hi = 6.5, 7.0

v_mask    = (z_lc_all[seed] >= z_lo) & (z_lc_all[seed] < z_hi)
v_true_2d = v_true_all[seed][:, :, v_mask].mean(axis=2)

print(f"v_true_2d std after averaging: {v_true_2d.std():.4e}")
print(f"v_true_2d range: {v_true_2d.min():.4e} to {v_true_2d.max():.4e}")
print(f"n slices averaged: {v_mask.sum()}")

# compare single slice vs averaged
print(f"single slice std: {v_true_all[seed][:,:,v_mask][:,:,0].std():.4e}")

v_true_2d std after averaging: 9.9862e-18
v_true_2d range: -2.9636e-17 to 2.0527e-17
n slices averaged: 14
single slice std: 2.3047e-17


In [7]:
# %%
# =============================================================================
# CELL 6: Seed-average
# =============================================================================

print("\nCELL 6 — Seed-averaging")

all_z_labels = sorted({zl for s in RANDOM_SEEDS for zl in results[s]})
avg = {}

for zl in all_z_labels:
    seeds_with_data = [s for s in RANDOM_SEEDS if zl in results[s]]
    if not seeds_with_data:
        continue

    C_rec_list   = [results[s][zl]['C_rec']   for s in seeds_with_data]
    C_true_list  = [results[s][zl]['C_true']  for s in seeds_with_data]
    D_rec_list   = [results[s][zl]['D_rec']   for s in seeds_with_data]
    D_true_list  = [results[s][zl]['D_true']  for s in seeds_with_data]
    r_list       = [results[s][zl]['r']       for s in seeds_with_data]
    k_ref        = results[seeds_with_data[0]][zl]['k']
    z_center     = results[seeds_with_data[0]][zl]['z_center']

    avg[zl] = {
        'k':           k_ref,
        'z_center':    z_center,
        'n_seeds':     len(seeds_with_data),
        'D_rec_mean':  np.nanmean(D_rec_list,  axis=0),
        'D_rec_std':   np.nanstd (D_rec_list,  axis=0),
        'D_true_mean': np.nanmean(D_true_list, axis=0),
        'D_true_std':  np.nanstd (D_true_list, axis=0),
        'r_mean':      np.nanmean(r_list, axis=0),
        'r_std':       np.nanstd (r_list, axis=0),
    }
    print(f"  {zl}: {len(seeds_with_data)} seeds  "
          f"z_center={z_center:.2f}")

print("✓ Cell 6 done")



CELL 6 — Seed-averaging
  z5.0-5.5: 5 seeds  z_center=5.25
  z5.5-6.0: 5 seeds  z_center=5.75
  z6.0-6.5: 5 seeds  z_center=6.25
  z6.5-7.0: 5 seeds  z_center=6.75
  z7.0-7.5: 5 seeds  z_center=7.25
✓ Cell 6 done


/var/tmp/pbs.1575744.swarm/ipykernel_77976/706581984.py:32: RuntimeWarning: Mean of empty slice
  'r_mean':      np.nanmean(r_list, axis=0),
/user1/swanith/miniconda3/envs/p21c_v41/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1997: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


In [8]:
# %%
# =============================================================================
# CELL 7: PLOT 1 — C_ell vs k per z-bin  (rec vs true)
# =============================================================================

cmap = plt.cm.plasma
norm = mpl.colors.Normalize(vmin=5.0, vmax=8.0)

fig, axes = plt.subplots(2, 3, figsize=(15, 9), constrained_layout=True)
axes = axes.flatten()

for ax, zl in zip(axes, all_z_labels):
    if zl not in avg:
        ax.set_visible(False)
        continue
    res   = avg[zl]
    color = cmap(norm(res['z_center']))
    k     = res['k']

    ax.plot(k, np.abs(res['D_true_mean']),
            color=color, lw=2.5, ls='-',
            label=r'$v_{\rm true}$  (ceiling)')
    ax.fill_between(k,
        np.abs(res['D_true_mean']) - res['D_true_std'],
        np.abs(res['D_true_mean']) + res['D_true_std'],
        color=color, alpha=0.15)

    ax.plot(k, np.abs(res['D_rec_mean']),
            color=color, lw=2.0, ls='--',
            label=r'$v_{\rm rec}$  (LAE recon)')
    ax.fill_between(k,
        np.abs(res['D_rec_mean']) - res['D_rec_std'],
        np.abs(res['D_rec_mean']) + res['D_rec_std'],
        color=color, alpha=0.10)

    ax.set_xscale('log')
    ax.set_yscale('symlog', linthresh=1e-20)
    ax.set_xlabel(r'$k$  [cMpc$^{-1}$]')
    ax.set_ylabel(r'$D_k \equiv \frac{k^2}{2\pi} C_k$')
    ax.set_title(rf'$z={res["z_center"]:.2f}$  ({res["n_seeds"]} seeds)',
                 fontsize=12)
    ax.legend(fontsize=10)

for ax in axes[len(all_z_labels):]:
    ax.set_visible(False)

sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
fig.colorbar(sm, ax=axes.tolist(), label='Redshift $z$', shrink=0.6)

fig.suptitle(
    r'$D_k(T_{\rm kSZ} \times v)$  per redshift bin — rec (dashed) vs true (solid)',
    fontsize=14, fontweight='bold')

fig.savefig(f"{PLOT_DIR}/plot1_Dk_vs_k_per_zbin.png")
fig.savefig(f"{PLOT_DIR}/plot1_Dk_vs_k_per_zbin.pdf")
plt.close(fig)
print("✓ Plot 1 saved")


✓ Plot 1 saved


In [9]:

# %%
# =============================================================================
# CELL 8: PLOT 2 — Reconstruction correlation r(k) per z-bin
# =============================================================================

fig, ax = plt.subplots(figsize=(10, 7), constrained_layout=True)

for zl in all_z_labels:
    if zl not in avg:
        continue
    res   = avg[zl]
    color = cmap(norm(res['z_center']))
    valid = np.isfinite(res['r_mean']) & (res['r_mean'] > -2.0)

    ax.plot(res['k'][valid], res['r_mean'][valid],
            color=color, lw=2.0, marker='o', ms=4,
            label=rf"$z={res['z_center']:.2f}$")
    ax.fill_between(
        res['k'][valid],
        (res['r_mean'] - res['r_std'])[valid],
        (res['r_mean'] + res['r_std'])[valid],
        color=color, alpha=0.15)

ax.axhline(0, color='black', ls='--', lw=0.8, alpha=0.5)
ax.axhline(1, color='gray',  ls=':',  lw=0.8, alpha=0.5,
           label='perfect reconstruction')
ax.set_xscale('log')
ax.set_ylim(-0.5, 1.3)
ax.set_xlabel(r'$k$  [cMpc$^{-1}$]')
ax.set_ylabel(r'$r(k) = C_k^{\rm cross}/\sqrt{C_k^{\rm rec}\,C_k^{\rm true}}$')
ax.set_title('Velocity reconstruction correlation — LAEs at EoR',
             fontweight='bold')

sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
fig.colorbar(sm, ax=ax, label='Redshift $z$', pad=0.02)

fig.savefig(f"{PLOT_DIR}/plot2_r_vs_k.png")
fig.savefig(f"{PLOT_DIR}/plot2_r_vs_k.pdf")
plt.close(fig)
print("✓ Plot 2 saved")


✓ Plot 2 saved


In [10]:

# %%
# =============================================================================
# CELL 9: PLOT 3 — D_k vs z at fixed k  (reionization history view)
# =============================================================================

k_targets  = [0.02, 0.05, 0.10]   # cMpc^-1
colors_k   = ['navy', 'forestgreen', 'firebrick']
labels_k   = [r'$k=0.02$ cMpc$^{-1}$ (large scale)',
               r'$k=0.05$ cMpc$^{-1}$ (bubble scale)',
               r'$k=0.10$ cMpc$^{-1}$ (small scale)']

fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)

for ax, key, title in zip(
    axes,
    ['D_true_mean', 'D_rec_mean'],
    [r'$v_{\rm true}$ — upper bound',
     r'$v_{\rm rec}$ — reconstructed from LAEs']
):
    for k_t, col, lab in zip(k_targets, colors_k, labels_k):
        z_pts, D_pts, D_err = [], [], []
        for zl in all_z_labels:
            if zl not in avg:
                continue
            res = avg[zl]
            idx = int(np.argmin(np.abs(res['k'] - k_t)))
            D_val = res[key][idx]
            D_e   = res[key.replace('mean', 'std')][idx]
            if np.isfinite(D_val):
                z_pts.append(res['z_center'])
                D_pts.append(np.abs(D_val))
                D_err.append(D_e)

        if len(z_pts) > 1:
            z_pts = np.array(z_pts)
            D_pts = np.array(D_pts)
            D_err = np.array(D_err)
            ax.errorbar(z_pts, D_pts, yerr=D_err,
                        color=col, lw=2.0, marker='o', ms=6,
                        capsize=3, label=lab)

    ax.set_xlabel(r'Redshift $z$')
    ax.set_ylabel(r'$|D_k|$')
    ax.set_yscale('symlog', linthresh=1e-22)
    ax.set_title(title, fontweight='bold')
    ax.invert_xaxis()
    ax.legend(fontsize=11)

fig.suptitle(r'$D_k(T_{\rm kSZ} \times v)$ vs redshift — reionization history',
             fontsize=14, fontweight='bold')

fig.savefig(f"{PLOT_DIR}/plot3_Dk_vs_z.png")
fig.savefig(f"{PLOT_DIR}/plot3_Dk_vs_z.pdf")
plt.close(fig)
print("✓ Plot 3 saved")

✓ Plot 3 saved


In [11]:
# %%
# =============================================================================
# CELL 10: Summary table
# =============================================================================

print("\n" + "="*70)
print("SUMMARY: Reconstruction penalty r per z-bin")
print("(SNR_realistic / SNR_true ≈ r²)")
print("="*70)
print(f"{'z-bin':<18} {'z_c':>6} {'r_med':>8} {'r_std':>8} {'seeds':>6}")
print("-"*50)

for zl in all_z_labels:
    if zl not in avg:
        continue
    res   = avg[zl]
    r_med = float(np.nanmedian(res['r_mean']))
    r_std = float(np.nanmedian(res['r_std']))
    print(f"{zl:<18} {res['z_center']:>6.2f} {r_med:>8.3f} "
          f"{r_std:>8.3f} {res['n_seeds']:>6d}")

print("="*70)
print("Plots saved to:", PLOT_DIR)
print("Outputs at:   ", CACHE_DIR_WRITE)


SUMMARY: Reconstruction penalty r per z-bin
(SNR_realistic / SNR_true ≈ r²)
z-bin                 z_c    r_med    r_std  seeds
--------------------------------------------------
z5.0-5.5             5.25   -0.159    0.215      5
z5.5-6.0             5.75   -0.014    0.181      5
z6.0-6.5             6.25   -0.075    0.192      5
z6.5-7.0             6.75    0.088    0.186      5
z7.0-7.5             7.25   -0.034    0.189      5
Plots saved to: /user1/swanith/kSZ2_halo_project/ksz_vrec/plots
Outputs at:    /user1/swanith/kSZ2_halo_project/ksz_vrec
